In [1]:
import os

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
if os.environ["OPENAI_API_KEY"]:
    print("API key is set")

API key is set


In [3]:
print(os.getenv("OPENAI_API_KEY") is not None)

True


In [4]:
from langchain_openai import ChatOpenAI

In [5]:
llm = ChatOpenAI(model="gpt-5-nano",temperature=0)

In [6]:
response = llm.invoke("Hello, explain RAG in one sentence.")
print(response.content)

RAG (Retrieval-Augmented Generation) is a framework that retrieves relevant documents from a large corpus and then uses those documents to guide a text generator to produce a grounded answer.


In [7]:
response = llm.invoke("Tell me what is AI in one line")
print(response)
print(response.content)

content='Artificial intelligence is the field of computer science focused on creating systems that can perform tasks that normally require human intelligence, such as learning, reasoning, and problem solving.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 361, 'prompt_tokens': 14, 'total_tokens': 375, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ELE8Km4qw7diW77LzPEt5oICTg5Uu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0787e-1f75-7ff2-8529-270b00b2a787-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens

# RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA


### STEP 1: Extracting text from pdf 

In [8]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/NISM-15.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

C:\Users\harsh\AppData\Local\Temp\ipykernel_4884\1167517523.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
docs[0]

Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-09T15:03:04+05:30', 'author': 'Rohit Jain', 'moddate': '2026-04-09T15:03:04+05:30', 'source': './Docs/NISM-15.pdf', 'total_pages': 342, 'page': 0, 'page_label': '1'}, page_content='1')

In [10]:
docs

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-09T15:03:04+05:30', 'author': 'Rohit Jain', 'moddate': '2026-04-09T15:03:04+05:30', 'source': './Docs/NISM-15.pdf', 'total_pages': 342, 'page': 0, 'page_label': '1'}, page_content='1'),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-09T15:03:04+05:30', 'author': 'Rohit Jain', 'moddate': '2026-04-09T15:03:04+05:30', 'source': './Docs/NISM-15.pdf', 'total_pages': 342, 'page': 1, 'page_label': '2'}, page_content='2 \n \nWorkbook for \nNISM-Series-XV: Research Analyst Certification Examination  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNational Institute of Securities Markets  \nwww.nism.ac.in'),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-09T15:03:04+05:30', 'author': 'Rohit Jain', 'moddate': '2026-04-09T15:03:04+05:30'

In [11]:
len(docs)

342

### STEP 2: Splitting the Document into CHUNKS

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#### Creating own Metadata for PDF Chunks

In [13]:
docs[0]

Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-09T15:03:04+05:30', 'author': 'Rohit Jain', 'moddate': '2026-04-09T15:03:04+05:30', 'source': './Docs/NISM-15.pdf', 'total_pages': 342, 'page': 0, 'page_label': '1'}, page_content='1')

In [14]:
docs[0].metadata

{'producer': 'Microsoft® Word LTSC',
 'creator': 'Microsoft® Word LTSC',
 'creationdate': '2026-04-09T15:03:04+05:30',
 'author': 'Rohit Jain',
 'moddate': '2026-04-09T15:03:04+05:30',
 'source': './Docs/NISM-15.pdf',
 'total_pages': 342,
 'page': 0,
 'page_label': '1'}

In [15]:
for i in docs:
    i.metadata = {"source": "NISM-15.pdf",
                  "document_id": "15"}

In [16]:
docs[0].metadata

{'source': 'NISM-15.pdf', 'document_id': '15'}

In [17]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'NISM-15.pdf', 'document_id': '15'}, page_content='1'),
 Document(metadata={'source': 'NISM-15.pdf', 'document_id': '15'}, page_content='2 \n \nWorkbook for \nNISM-Series-XV: Research Analyst Certification Examination  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNational Institute of Securities Markets  \nwww.nism.ac.in'),
 Document(metadata={'source': 'NISM-15.pdf', 'document_id': '15'}, page_content='3 \n \nThis workbook has been developed to assist candidates in preparing for the National Institute of \nSecurities Markets (NISM) Certification Examination for Research Analyst ( NISM-Series-XV: Research \nAnalyst Certification Examination).   \n \n \n \nWorkbook Version: February 20261 \n \nPublished by: \nNational Institute of Securities Markets  \n© National Institute of Securities Markets, 2020 \nNISM Registered Office \n5th floor, NCL Cooperative Society, \nPlot No. C-6, E-Block, Bandra Kurla Complex, \nBandra East, Mumbai, 400051. \n \

In [18]:
len(chunks)

986

### STEP 3: Creating Embeddings for the Chunks

In [19]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")  ## model="text-embedding-3-small" is default

In [20]:
embedding_model.embed_query("What is nism?")

[-0.0277862548828125,
 0.038421630859375,
 -0.004810333251953125,
 0.00106048583984375,
 0.0133056640625,
 0.01395416259765625,
 -0.0032863616943359375,
 0.03277587890625,
 0.0023899078369140625,
 -0.025787353515625,
 0.012420654296875,
 -0.016326904296875,
 -0.073486328125,
 -0.03546142578125,
 0.0731201171875,
 0.0205230712890625,
 -0.056854248046875,
 0.0079345703125,
 0.0036334991455078125,
 0.0249786376953125,
 0.050811767578125,
 0.034332275390625,
 -0.041473388671875,
 0.0190887451171875,
 0.00197601318359375,
 -0.0164642333984375,
 -0.0179290771484375,
 -0.00897216796875,
 0.07952880859375,
 -0.0007672309875488281,
 0.072509765625,
 -0.03814697265625,
 0.01319122314453125,
 -0.0130767822265625,
 -0.002872467041015625,
 0.0115966796875,
 -0.001922607421875,
 -0.01513671875,
 -0.004638671875,
 -0.0028743743896484375,
 -0.048980712890625,
 -0.041656494140625,
 -0.0189361572265625,
 0.0303497314453125,
 -0.0279388427734375,
 0.007732391357421875,
 -0.0194244384765625,
 -0.013000488

In [21]:
vector = embedding_model.embed_query("What is nism?")
print(len(vector))


1536


In [22]:
vector = embedding_model.embed_query("What is Research Analyst?")
print(len(vector))


1536


### STEP 4: Create and Store Embeddings in Local Vector Store

In [23]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector_store/"
)

In [24]:
import chromadb
print(chromadb.__version__)

1.5.9


In [25]:
import sys
print(sys.executable)

import site
print(site.getsitepackages())


d:\rag-1-project\rag_venv\Scripts\python.exe
['d:\\rag-1-project\\rag_venv', 'd:\\rag-1-project\\rag_venv\\Lib\\site-packages']


In [26]:
# vectors = []
# for doc in chunks:
#     vector = embedding_model.embed_documents([doc.page_content])
#     vectors.append(vector)
# print(vectors)    
# print("Number of vectors:", len(vectors))
# print("Dimensions of first vector:", len(vectors[0]))


### STEP 5: Semantic Search

In [27]:
vectorstore.similarity_search("What is NISM?", k= 3)

[Document(metadata={'document_id': '15', 'source': 'NISM-15.pdf'}, page_content='intermediaries focusing on varied product lines and functional areas. NISM certifications  have \nestablished knowledge benchmarks for various market products and functions such as equities, mutual \nfunds, derivatives, compliance, operations, advisory and research. NISM certification examinations and \ntraining programs provide a structured learning  plan and career path to students and job aspirants, \nwishing to make a professional career in the securities markets.  \nNISM supports candidates by providing lucid and focused workbooks that assist them in understanding \nthe subject and preparing for NISM Ex aminations. This book covers all important topics required to \nundertake research on companies. These include the basics of Indian securities markets, various \nterminologies used in the equity and debt markets, top down and bottom up approach to fundamental \nresearch, basic principles for micro and 

### Talk to LLM

In [28]:
context = vectorstore.similarity_search("What is NISM?", k= 3)

In [29]:
response = llm.invoke(f"What is NISM? You can answer using the following context: {context}")
print(response.content)
print(10*"*")
print(response)

NISM stands for the National Institute of Securities Markets. It is a leading Indian institution that develops and administers professional education, certifications, training, and research in financial markets. NISM creates certification examinations and Continuing Professional Education (CPE) programs for securities market professionals to ensure a defined minimum common knowledge benchmark, as mandated under SEBI regulations (Certification of Associated Persons in the Securities Markets Regulations, 2007).
**********
content='NISM stands for the National Institute of Securities Markets. It is a leading Indian institution that develops and administers professional education, certifications, training, and research in financial markets. NISM creates certification examinations and Continuing Professional Education (CPE) programs for securities market professionals to ensure a defined minimum common knowledge benchmark, as mandated under SEBI regulations (Certification of Associated Pers

In [30]:
response = llm.invoke(f"What is agentic coding? You can answer using the following context: {context}")
print(response.content)
print(5*"*")
print(response)

The provided documents (NISM-15.pdf) do not mention or define "agentic coding." They discuss NISM certifications and continuing education for securities market professionals, but there is no reference to the term "agentic coding." If you have another source or can clarify what you mean by the term, I can help interpret it.
*****
content='The provided documents (NISM-15.pdf) do not mention or define "agentic coding." They discuss NISM certifications and continuing education for securities market professionals, but there is no reference to the term "agentic coding." If you have another source or can clarify what you mean by the term, I can help interpret it.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 652, 'prompt_tokens': 612, 'total_tokens': 1264, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audi

### Re-Use the Vector Database

In [31]:
vectorstore_persist = Chroma(
    persist_directory="./Vector_store/",
    embedding_function=embedding_model
)

C:\Users\harsh\AppData\Local\Temp\ipykernel_4884\3551676164.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(


In [33]:
vectorstore_persist.similarity_search("Why is primary role of RA?", k= 3)

[Document(metadata={'source': 'NISM-15.pdf', 'document_id': '15'}, page_content='21 \n \nSample Questions \n1. What is the role of Research Analyst? \na. RAs are only involved in the analysis of data \nb. RAs are only involved in collection of the data \nc. RAs help their clients take informed decisions \nd. RAs help in financial planning of their client'),
 Document(metadata={'document_id': '15', 'source': 'NISM-15.pdf'}, page_content='• Communication, done through written research reports, should be simple, clear and concise. \n• If there is any conflict of interest (e.g., RA holds shares of the subject company), such information \nshould be disclosed beforehand. \n• Assumptions, if any, must be clearly stated in the research reports. \n• Abbreviations/Jargons should either be avoided or explained clearly in simple words. \nThe role of RA s is to collect data/information from different reliable sources, interpret the \ndata/information and convert it into recommendations that their c

In [34]:
context = vectorstore_persist.similarity_search("What is ESG framework for company analysis", k= 3)

In [35]:
response = llm.invoke(f"What is ESG framework for company analysis? You can answer using the following context: {context}")
print(response.content)
print(10*"*")
print(response)

ESG framework is a set of non-financial criteria used to analyze a company based on Environmental, Social, and Governance factors. It helps investors assess potential risks and long-term value beyond traditional financial metrics.

- Environmental: how the company affects the environment (e.g., low carbon emissions, pollution control).
- Social: the company’s impact on society (e.g., human rights, gender equality, social development).
- Governance: the quality of the company’s governance practices (e.g., board standards, transparency, risk management).

Why it’s used:
- ESG factors can influence a company’s risk profile and cost of capital.
- Companies focused on environmental issues may face less regulatory disruption; those strong in social factors can have better recruitment and customer appeal; robust governance can reduce risk perception.

How it’s used in analysis:
- Equity analysts discuss ESG parameters to guide investors who care about these factors.
- Investors may use ESG cr